# Cell 1 — Import **Libraries**

In [1]:

# Lab 2.1 - Step 1: Data Preparation

import pandas as pd
import numpy as np
import hashlib
import sys

SEED = 42

print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("Random seed:", SEED)


Python version: 3.13.18
Pandas version: 2.2.3
Random seed: 42


# **Cell 2 — Upload Dataset**

In [2]:

from google.colab import files

uploaded = files.upload()


KeyboardInterrupt: 

# **Cell 3 — Load and Inspect Data**

In [3]:

# Load dataset
df = pd.read_csv(
    "COE67-342_W01_Lab_RestaurantReviews.csv"
)

# Load golden reference
gold = pd.read_csv(
    "COE67-342_W02_Gold_WordSeg_3000.tsv",
    sep="\t"
)

print("Total reviews:", len(df))
print("Golden references:", len(gold))

print("\nDataset preview:")
display(df.head())

print("\nGolden Set preview:")
display(gold.head())


Total reviews: 100000
Golden references: 3000

Dataset preview:


,review_id,text,label
0,R000001,เป็นลูกค้ามา 8 ปี แสงสวย ใครมาพัทยาต้องแวะ ฮะ,pos
1,R000002,มากับแก๊งเพื่อนมหาลัย เจ้าของร้านคุยกันเสียงดั...,neg
2,R000003,มาทำธุระแถวถนนคนเดิน มีเจลล้างมือทุกโต๊ะ แต่ซา...,neu
3,R000004,ขับรถมาจากแม่ริมเพื่อร้านนี้ ทอดไส้กรอกกับซุปเ...,pos
4,R000005,หัวหน้าพามา น้ำจิ้มเข้มข้นมาก เจ้าของร้านช่วยถ...,pos



Golden Set preview:


,review_id,text,tokens,typo_manual
0,R075722,พาเพื่อนสมัยมัธยมมาฉลองวันครบรอบ ทอดกระเทียมเน...,พา|เพื่อน|สมัย|มัธยม|มา|ฉลอง|วัน|ครบรอบ|ทอด|กร...,0
1,R080185,มากินครั้งแรก คิดเงินเกินไป 70 บาท ติดตรงที่อบ...,มา|กิน|ครั้ง|แรก|คิด|เงิน|เกิน|ไป|70|บาท|ติด|ต...,0
2,R019865,ตามรีวิวจากแผนที่ออนไลน์มา คนคิดเงินเรียกไม่มา...,ตาม|รีวิว|จาก|แผนที่|ออนไลน์|มา|คน|คิด|เงิน|เร...,0
3,R076700,มากับแขกต่างชาติ ราคาประมาณ 840 บาทต่อคน ต้องจ...,มา|กับ|แขก|ต่างชาติ|ราคา|ประมาณ|840|บาท|ต่อ|คน...,0
4,R092992,เห็นจากยูทูบ เกาเหลาไม่ถึงเครื่อง ไม่อร่อย ติด...,เห็น|จาก|ยูทูบ|เกาเหลา|ไม่|ถึง|เครื่อง|ไม่|อร่...,0


# **Cell 4 — Sample 20,000 Reviews**

In [4]:

# Sample 20,000 reviews
sampled_df = df.sample(
    n=20000,
    random_state=SEED
).copy()

# Sort by review ID for consistent file order
sampled_df = sampled_df.sort_values(
    "review_id"
).reset_index(drop=True)

print("Sampled reviews:", len(sampled_df))

display(sampled_df.head())


Sampled reviews: 20000


,review_id,text,label
0,R000009,ตาเอ๋แถวสตูล เจ้าของร้านลืมออเดอร์ คนรับรถเลือ...,neg
1,R000024,ครัวคุณโบว์แถวอุดรธานี กลับมากิงติ่มซำอีกรอบ ก...,pos
2,R000040,ตั้งใจขับมาจากอุตรดิตถ์ ผัดเห็ดเข็มทองกุ้งจานเ...,neg
3,R000054,ร้านนี้ดีจริง ดนตรีสดเพราะ ตุ๋นเครื่องในรสมื...,pos
4,R000071,ผ่านมาเลยลองแวะ จ่ายไป 650 บาท คุ้มจังงงงงง เส...,neu


# **Cell 5 — Freeze Golden Evaluation Set**

In [5]:

# Check required columns
assert {"review_id", "text"}.issubset(df.columns)
assert {"review_id", "text", "tokens"}.issubset(gold.columns)

# Check unique IDs
assert df["review_id"].is_unique
assert gold["review_id"].is_unique

# Check golden IDs in sampled dataset
eval_df = gold.merge(
    sampled_df[["review_id", "text"]],
    on="review_id",
    how="left",
    suffixes=("_gold", "_sample"),
    validate="one_to_one",
    indicator=True
)

assert (eval_df["_merge"] == "both").all()
assert (
    eval_df["text_gold"] == eval_df["text_sample"]
).all()

# Save frozen evaluation references
frozen_eval = gold.sort_values(
    "review_id"
).reset_index(drop=True)

frozen_eval.to_csv(
    "frozen_eval.csv",
    index=False,
    encoding="utf-8-sig"
)

# Calculate SHA256
with open("frozen_eval.csv", "rb") as f:
    file_hash = hashlib.sha256(f.read()).hexdigest()

print("Frozen evaluation:", len(frozen_eval))
print("SHA256:", file_hash)
print("Saved: frozen_eval.csv")


Frozen evaluation: 3000
SHA256: 9bfb8762bf4c0cf00cd3c2d44da1eeed2fee79853f236e13232f6d81dbfdb2b0
Saved: frozen_eval.csv


# **Cell 6 — Save Sampled Dataset**

In [6]:

sampled_df.to_csv(
    "sampled_20000.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: sampled_20000.csv")
print("Total rows:", len(sampled_df))


Saved: sampled_20000.csv
Total rows: 20000


Cell 6.1

In [7]:

# Lab 2.1 - Experiment Settings

print("Experiment: Lab 2.1")
print("Random Seed:", SEED)
print("Total Dataset:", len(df))
print("Sampled Dataset:", len(sampled_df))
print("Golden Evaluation:", len(frozen_eval))
print("Normalization: normalize_minimal")
print("Tokenizers: newmm, longest, attacut")


Experiment: Lab 2.1
Random Seed: 42
Total Dataset: 100000
Sampled Dataset: 20000
Golden Evaluation: 3000
Normalization: normalize_minimal
Tokenizers: newmm, longest, attacut


# **Cell 7 — Create Normalization Function**

In [8]:

# Cell 7: Minimal Text Normalization

import re
import unicodedata

def normalize_minimal(text):
    # 1. Unicode NFC
    text = unicodedata.normalize("NFC", str(text))

    # 2. Remove zero-width characters
    text = re.sub(r"[\u200B\u200C\u200D\uFEFF]", "", text)

    # 3. Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

print("Normalization function created successfully!")


Normalization function created successfully!


# **Cell 8 — Test Normalization**

In [9]:

# Cell 8: Test Normalization

test_texts = [
    "อาหาร   อร่อยมาก",
    "ฉันไม่ชอบร้านนี้",
    "อร่อยยยยมากกก",
    "วันนี้\u200Bอากาศดี",
    "บริการดี\n\nมาก"
]

for text in test_texts:
    result = normalize_minimal(text)

    print("Before:", repr(text))
    print("After: ", repr(result))
    print("-" * 40)


Before: 'อาหาร   อร่อยมาก'
After:  'อาหาร อร่อยมาก'
----------------------------------------
Before: 'ฉันไม่ชอบร้านนี้'
After:  'ฉันไม่ชอบร้านนี้'
----------------------------------------
Before: 'อร่อยยยยมากกก'
After:  'อร่อยยยยมากกก'
----------------------------------------
Before: 'วันนี้\u200bอากาศดี'
After:  'วันนี้อากาศดี'
----------------------------------------
Before: 'บริการดี\n\nมาก'
After:  'บริการดี มาก'
----------------------------------------


# **Cell 9 — Apply Normalization to Dataset**

In [10]:

# Cell 9: Apply Normalization

# Normalize 20,000 reviews
sampled_df["normalized_text"] = (
    sampled_df["text"].apply(normalize_minimal)
)

# Normalize 3,000 golden references
frozen_eval["normalized_text"] = (
    frozen_eval["text"].apply(normalize_minimal)
)

print("Normalized sampled reviews:", len(sampled_df))
print("Normalized evaluation reviews:", len(frozen_eval))

# Show sample results
display(
    sampled_df[
        ["review_id", "text", "normalized_text"]
    ].head()
)


Normalized sampled reviews: 20000
Normalized evaluation reviews: 3000


,review_id,text,normalized_text
0,R000009,ตาเอ๋แถวสตูล เจ้าของร้านลืมออเดอร์ คนรับรถเลือ...,ตาเอ๋แถวสตูล เจ้าของร้านลืมออเดอร์ คนรับรถเลือ...
1,R000024,ครัวคุณโบว์แถวอุดรธานี กลับมากิงติ่มซำอีกรอบ ก...,ครัวคุณโบว์แถวอุดรธานี กลับมากิงติ่มซำอีกรอบ ก...
2,R000040,ตั้งใจขับมาจากอุตรดิตถ์ ผัดเห็ดเข็มทองกุ้งจานเ...,ตั้งใจขับมาจากอุตรดิตถ์ ผัดเห็ดเข็มทองกุ้งจานเ...
3,R000054,ร้านนี้ดีจริง ดนตรีสดเพราะ ตุ๋นเครื่องในรสมื...,ร้านนี้ดีจริง ดนตรีสดเพราะ ตุ๋นเครื่องในรสมือด...
4,R000071,ผ่านมาเลยลองแวะ จ่ายไป 650 บาท คุ้มจังงงงงง เส...,ผ่านมาเลยลองแวะ จ่ายไป 650 บาท คุ้มจังงงงงง เส...


# **Cell 10 — Install Libraries**

In [12]:

# Cell 10: Install Thai Tokenizers

!pip install pythainlp==5.4.0 attacut==1.0.6


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 95.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 90.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.3 MB/s  0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=fe741ce234b645382aaa7816f2ed5a8fc0f638a3b9ebc901940472a1681dba00
  Stored in directory: /root/.cache/pip/wheels/0b/1d/03/175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
  Created wheel for numpy: filename=numpy-1.26.4-cp313-cp313-linux_x86_64.whl size=7039804 sha256=397af51b14b2820359acd4a1ee

# **Cell 11 — Test Three Tokenizers**

In [11]:

# Cell 11: Test Thai Word Tokenization

from pythainlp.tokenize import word_tokenize

text = "อาหารอร่อยมากแต่บริการไม่ดี"
text = normalize_minimal(text)

tokens_newmm = word_tokenize(
    text,
    engine="newmm",
    keep_whitespace=False
)

tokens_longest = word_tokenize(
    text,
    engine="longest",
    keep_whitespace=False
)

tokens_attacut = word_tokenize(
    text,
    engine="attacut",
    keep_whitespace=False
)

print("Original:", text)
print("newmm:   ", tokens_newmm)
print("longest: ", tokens_longest)
print("attacut: ", tokens_attacut)


Original: อาหารอร่อยมากแต่บริการไม่ดี
newmm:    ['อาหาร', 'อร่อย', 'มาก', 'แต่', 'บริการ', 'ไม่', 'ดี']
longest:  ['อาหาร', 'อร่อย', 'มาก', 'แต่', 'บริการ', 'ไม่', 'ดี']
attacut:  ['อาหาร', 'อร่อย', 'มาก', 'แต่', 'บริการ', 'ไม่', 'ดี']


# **Cell 12 — Create Three Tokenization Pipelines**

In [12]:

# Cell 12: Create Tokenization Pipelines

from pythainlp.tokenize import word_tokenize

def tokenize_text(text, engine):
    return word_tokenize(
        text,
        engine=engine,
        keep_whitespace=False
    )

pipelines = {
    "A": "newmm",
    "B": "longest",
    "C": "attacut"
}

print("Pipelines:", pipelines)


Pipelines: {'A': 'newmm', 'B': 'longest', 'C': 'attacut'}


# **Cell 13 — Run Three Tokenizers on 20,000 Reviews**

In [13]:

# Cell 13: Run Tokenization Experiment

import time

results = {}
timing_results = []

for name, engine in pipelines.items():
    print(f"Running Pipeline {name}: {engine}")

    start = time.perf_counter()

    tokens = sampled_df["normalized_text"].apply(
        lambda text: tokenize_text(text, engine)
    )

    elapsed = time.perf_counter() - start

    results[name] = tokens

    timing_results.append({
        "pipeline": name,
        "engine": engine,
        "time_seconds": elapsed,
        "total_tokens": tokens.apply(len).sum()
    })

    print(f"Completed in {elapsed:.2f} seconds\n")

timing_df = pd.DataFrame(timing_results)

display(timing_df)


Running Pipeline A: newmm
Completed in 5.58 seconds

Running Pipeline B: longest
Completed in 57.22 seconds

Running Pipeline C: attacut
Completed in 85.48 seconds



,pipeline,engine,time_seconds,total_tokens
0,A,newmm,5.579437,477094
1,B,longest,57.219711,482651
2,C,attacut,85.475114,502928


# **Cell 14 — Compare Tokenization Outputs**

In [14]:

# Cell 14: Compare Tokenization Outputs

comparison_df = sampled_df[
    ["review_id", "normalized_text"]
].copy()

for name in pipelines:
    comparison_df[f"tokens_{name}"] = results[name]

display(comparison_df.head(10))


,review_id,normalized_text,tokens_A,tokens_B,tokens_C
0,R000009,ตาเอ๋แถวสตูล เจ้าของร้านลืมออเดอร์ คนรับรถเลือ...,"[ตา, เอ๋, แถว, สตูล, เจ้าของร้าน, ลืม, ออเดอร์...","[ตา, เอ๋, แถว, สตูล, เจ้าของร้าน, ลืม, ออเดอร์...","[ตา, เอ๋, แถว, สตูล, เจ้าของ, ร้าน, ลืม, ออเดอ..."
1,R000024,ครัวคุณโบว์แถวอุดรธานี กลับมากิงติ่มซำอีกรอบ ก...,"[ครัว, คุณ, โบว์, แถว, อุดรธานี, กลับมา, กิง, ...","[ครัว, คุณ, โบว์, แถว, อุดรธานี, กลับ, มากิง, ...","[ครัว, คุณ, โบว์, แถว, อุดรธานี, กลับ, มา, กิง..."
2,R000040,ตั้งใจขับมาจากอุตรดิตถ์ ผัดเห็ดเข็มทองกุ้งจานเ...,"[ตั้งใจ, ขับ, มาจาก, อุตรดิตถ์, ผัด, เห็ด, เข็...","[ตั้งใจ, ขับ, มาจาก, อุตรดิตถ์, ผัด, เห็ด, เข็...","[ตั้งใจ, ขับ, มา, จาก, อุตรดิตถ์, ผัด, เห็ดเข็..."
3,R000054,ร้านนี้ดีจริง ดนตรีสดเพราะ ตุ๋นเครื่องในรสมือด...,"[ร้าน, นี้, ดีจริง, ดนตรี, สด, เพราะ, ตุ๋น, เค...","[ร้าน, นี้, ดีจริง, ดนตรี, สด, เพราะ, ตุ๋น, เค...","[ร้าน, นี้, ดี, จริง, ดนตรี, สด, เพราะ, ตุ๋น, ..."
4,R000071,ผ่านมาเลยลองแวะ จ่ายไป 650 บาท คุ้มจังงงงงง เส...,"[ผ่าน, มา, เลย, ลอง, แวะ, จ่าย, ไป, 650, บาท, ...","[ผ่าน, มา, เลย, ลอง, แวะ, จ่าย, ไป, 650, บาท, ...","[ผ่าน, มา, เลย, ลอง, แวะ, จ่าย, ไป, 650, บาท, ..."
5,R000076,มากับคู่เดต มาถึงได้โต๊ะเลย ต้องบอกว่าลมพัดเย็...,"[มา, กับ, คู่, เดต, มาถึง, ได้, โต๊ะ, เลย, ต้อ...","[มา, กับ, คู่, เดต, มาถึง, ได้, โต๊ะ, เลย, ต้อ...","[มา, กับ, คู่เดต, มา, ถึง, ได้, โต๊ะ, เลย, ต้อ..."
6,R000088,ร้านนี้ดีจริง ผัดฉ่าเบคอนรสชาติเหมือนที่คุณแม่...,"[ร้าน, นี้, ดีจริง, ผัดฉ่า, เบคอน, รสชาติ, เหม...","[ร้าน, นี้, ดีจริง, ผัดฉ่า, เบคอน, รสชาติ, เหม...","[ร้าน, นี้, ดี, จริง, ผัด, ฉ่าเบคอนรสชาติ, เหม..."
7,R000089,ร้านประจำ ยำทะเลหอมมันสุด ๆ ส่งไว ของยังร้อน พ...,"[ร้าน, ประจำ, ยำ, ทะเล, หอม, มัน, สุด, ๆ, ส่ง,...","[ร้าน, ประจำ, ยำ, ทะเล, หอม, มัน, สุด, ๆ, ส่ง,...","[ร้าน, ประจำ, ยำ, ทะเล, หอมมัน, สุด, ๆ, ส่ง, ไ..."
8,R000096,มาซ้ำรอบที่ 9 แล้ว มีมุมเด็กเล่น น้ำจิ้มรสจัดข...,"[มา, ซ้ำ, รอบ, ที่, 9, แล้ว, มี, มุม, เด็ก, เล...","[มา, ซ้ำ, รอบ, ที่, 9, แล้ว, มี, มุม, เด็ก, เล...","[มา, ซ้ำ, รอบ, ที่, 9, แล้ว, มี, มุม, เด็ก, เล..."
9,R000098,มากินครั้งแรก สั่งตุ๋นสันคอหมูดอกแคมา โดนใจจริ...,"[มา, กิน, ครั้งแรก, สั่ง, ตุ๋น, สัน, คอ, หมู, ...","[มา, กิน, ครั้งแรก, สั่ง, ตุ๋น, สัน, คอ, หมู, ...","[มา, กิน, ครั้ง, แรก, สั่ง, ตุ๋น, สัน, คอ, หมู..."


# **Cell 15 — ตรวจสอบชุด Golden Set**

In [15]:

# Cell 15: Inspect Golden Reference Tokens

print("Golden evaluation size:", len(frozen_eval))

display(
    frozen_eval[
        ["review_id", "text", "tokens", "normalized_text"]
    ].head(5)
)

# Check whether reference tokens reconstruct normalized text
def check_reference(row):
    reference_tokens = str(row["tokens"]).split("|")
    reconstructed = "".join(reference_tokens)
    return normalize_minimal(reconstructed) == row["normalized_text"]

frozen_eval["reference_match"] = frozen_eval.apply(
    check_reference,
    axis=1
)

print("Matched references:",
      frozen_eval["reference_match"].sum())

print("Unmatched references:",
      (~frozen_eval["reference_match"]).sum())

display(
    frozen_eval.loc[
        ~frozen_eval["reference_match"],
        ["review_id", "text", "tokens"]
    ].head(5)
)


Golden evaluation size: 3000


,review_id,text,tokens,normalized_text
0,R000088,ร้านนี้ดีจริง ผัดฉ่าเบคอนรสชาติเหมือนที่คุณแม่...,ร้าน|นี้|ดี|จริง|ผัดฉ่า|เบคอน|รสชาติ|เหมือน|ที...,ร้านนี้ดีจริง ผัดฉ่าเบคอนรสชาติเหมือนที่คุณแม่...
1,R000098,มากินครั้งแรก สั่งตุ๋นสันคอหมูดอกแคมา โดนใจจริ...,มา|กิน|ครั้ง|แรก|สั่ง|ตุ๋น|สันคอหมู|ดอก|แค|มา|...,มากินครั้งแรก สั่งตุ๋นสันคอหมูดอกแคมา โดนใจจริ...
2,R000115,พาเพื่อนร่วมงานมาฉลองปีใหม่ บัวลอยจานละ 410 บา...,พา|เพื่อนร่วมงาน|มา|ฉลอง|ปีใหม่|บัวลอย|จาน|ละ|...,พาเพื่อนร่วมงานมาฉลองปีใหม่ บัวลอยจานละ 410 บา...
3,R000128,แวะมาตอนไปอุดรธานี มาถึงได้โต๊ะเลย พูดตรง ๆ ผั...,แวะ|มา|ตอน|ไป|อุดรธานี|มา|ถึง|ได้|โต๊ะ|เลย|พูด...,แวะมาตอนไปอุดรธานี มาถึงได้โต๊ะเลย พูดตรง ๆ ผั...
4,R000135,ไม่ประทับใจเลย น้องมิ้นท์วางจานกระแทก แย่มาก ส...,ไม่|ประทับใจ|เลย|น้อง|มิ้นท์|วาง|จาน|กระแทก|แย...,ไม่ประทับใจเลย น้องมิ้นท์วางจานกระแทก แย่มาก ส...


Matched references: 60
Unmatched references: 2940


,review_id,text,tokens
0,R000088,ร้านนี้ดีจริง ผัดฉ่าเบคอนรสชาติเหมือนที่คุณแม่...,ร้าน|นี้|ดี|จริง|ผัดฉ่า|เบคอน|รสชาติ|เหมือน|ที...
1,R000098,มากินครั้งแรก สั่งตุ๋นสันคอหมูดอกแคมา โดนใจจริ...,มา|กิน|ครั้ง|แรก|สั่ง|ตุ๋น|สันคอหมู|ดอก|แค|มา|...
2,R000115,พาเพื่อนร่วมงานมาฉลองปีใหม่ บัวลอยจานละ 410 บา...,พา|เพื่อนร่วมงาน|มา|ฉลอง|ปีใหม่|บัวลอย|จาน|ละ|...
3,R000128,แวะมาตอนไปอุดรธานี มาถึงได้โต๊ะเลย พูดตรง ๆ ผั...,แวะ|มา|ตอน|ไป|อุดรธานี|มา|ถึง|ได้|โต๊ะ|เลย|พูด...
4,R000135,ไม่ประทับใจเลย น้องมิ้นท์วางจานกระแทก แย่มาก ส...,ไม่|ประทับใจ|เลย|น้อง|มิ้นท์|วาง|จาน|กระแทก|แย...


# **Cell 16 — ตรวจสอบโดยไม่นับช่องว่าง**

In [16]:

# Cell 16: Check Golden References Without Spaces

def remove_spaces(text):
    return "".join(str(text).split())

def check_reference_no_space(row):
    reference_tokens = str(row["tokens"]).split("|")
    reconstructed = "".join(reference_tokens)

    reference = remove_spaces(
        normalize_minimal(reconstructed)
    )

    original = remove_spaces(
        row["normalized_text"]
    )

    return reference == original

frozen_eval["reference_match_no_space"] = (
    frozen_eval.apply(check_reference_no_space, axis=1)
)

print("Golden evaluation size:", len(frozen_eval))

print(
    "Matched (ignoring spaces):",
    frozen_eval["reference_match_no_space"].sum()
)

print(
    "Unmatched (ignoring spaces):",
    (~frozen_eval["reference_match_no_space"]).sum()
)

display(
    frozen_eval.loc[
        ~frozen_eval["reference_match_no_space"],
        ["review_id", "text", "tokens"]
    ].head(5)
)


Golden evaluation size: 3000
Matched (ignoring spaces): 3000
Unmatched (ignoring spaces): 0


,review_id,text,tokens


# **Cell 17 — Create Boundary Evaluation Functions**

In [17]:

# Cell 17: Boundary Evaluation Functions

def get_boundaries(tokens):
    boundaries = set()
    position = 0

    for token in tokens:
        token = remove_spaces(token)
        position += len(token)
        boundaries.add(position)

    boundaries.discard(position)
    return boundaries


def calculate_boundary_metrics(reference, predicted):
    ref_boundaries = get_boundaries(reference)
    pred_boundaries = get_boundaries(predicted)

    tp = len(ref_boundaries & pred_boundaries)
    fp = len(pred_boundaries - ref_boundaries)
    fn = len(ref_boundaries - pred_boundaries)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0
    )

    return precision, recall, f1, tp, fp, fn


# Test
reference = ["ฉัน", "ชอบ", "กิน", "ข้าว"]
predicted = ["ฉัน", "ชอบกิน", "ข้าว"]

print(calculate_boundary_metrics(reference, predicted))


(1.0, 0.6666666666666666, 0.8, 2, 0, 1)


# **Cell 18 — Evaluate Three Pipelines on Golden Set**

In [21]:

# Cell 18: Evaluate Three Pipelines (Fixed)

evaluation_results = []

for name, engine in pipelines.items():
    print(f"Evaluating Pipeline {name}: {engine}")

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for _, row in frozen_eval.iterrows():
        text = row["normalized_text"]

        reference = str(row["tokens"]).split("|")
        predicted = tokenize_text(text, engine)

        # Check alignment (ignore uppercase/lowercase)
        ref_text = remove_spaces("".join(reference))
        pred_text = remove_spaces("".join(predicted))

        assert (
            len(ref_text) == len(pred_text)
            and ref_text.lower() == pred_text.lower()
        ), f"Alignment error: {row['review_id']}"

        # Calculate boundary metrics
        _, _, _, tp, fp, fn = calculate_boundary_metrics(
            reference, predicted
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = (
        total_tp / (total_tp + total_fp)
        if (total_tp + total_fp) else 0
    )

    recall = (
        total_tp / (total_tp + total_fn)
        if (total_tp + total_fn) else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0
    )

    evaluation_results.append({
        "pipeline": name,
        "engine": engine,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn
    })

metrics_df = pd.DataFrame(evaluation_results)

display(metrics_df)


Evaluating Pipeline A: newmm
Evaluating Pipeline B: longest
Evaluating Pipeline C: attacut


,pipeline,engine,precision,recall,f1,TP,FP,FN
0,A,newmm,0.953262,0.927235,0.940069,64390,3157,5053
1,B,longest,0.945619,0.930245,0.937869,64599,3715,4844
2,C,attacut,0.923225,0.950679,0.936751,66018,5490,3425


# **Cell 18.1**

In [19]:

# Cell 18.1: Debug Alignment Error

review = frozen_eval[
    frozen_eval["review_id"] == "R000929"
].iloc[0]

text = review["normalized_text"]

reference = str(review["tokens"]).split("|")
predicted = tokenize_text(text, "longest")

ref_text = remove_spaces("".join(reference))
pred_text = remove_spaces("".join(predicted))

print("Review ID:", review["review_id"])
print("\nOriginal text:")
print(repr(text))

print("\nGolden tokens:")
print(reference)

print("\nLongest tokens:")
print(predicted)

print("\nReference length:", len(ref_text))
print("Predicted length:", len(pred_text))
print("Same text:", ref_text == pred_text)

# Find the first different character
for i, (a, b) in enumerate(zip(ref_text, pred_text)):
    if a != b:
        print("\nFirst difference at position:", i)
        print("Reference:", repr(ref_text[max(0, i-15):i+20]))
        print("Predicted:", repr(pred_text[max(0, i-15):i+20]))
        break
else:
    if len(ref_text) != len(pred_text):
        print("\nTexts have different lengths.")
        print("Reference end:", repr(ref_text[-40:]))
        print("Predicted end:", repr(pred_text[-40:]))


Review ID: R000929

Original text:
'Juicy'

Golden tokens:
['Juicy']

Longest tokens:
['juicy']

Reference length: 5
Predicted length: 5
Same text: False

First difference at position: 0
Reference: 'Juicy'
Predicted: 'juicy'


# Cell 18.2 — **ตรวจสอบความแตกต่างของอักขระทั้งหมด**

In [20]:

# Cell 18.2: Check Alignment for All Pipelines

alignment_issues = []

for name, engine in pipelines.items():
    for _, row in frozen_eval.iterrows():

        reference = str(row["tokens"]).split("|")
        predicted = tokenize_text(
            row["normalized_text"], engine
        )

        ref_text = remove_spaces("".join(reference))
        pred_text = remove_spaces("".join(predicted))

        if ref_text != pred_text:
            alignment_issues.append({
                "review_id": row["review_id"],
                "pipeline": name,
                "engine": engine,
                "reference": ref_text,
                "predicted": pred_text,
                "case_only": (
                    ref_text.lower() == pred_text.lower()
                    and len(ref_text) == len(pred_text)
                )
            })

alignment_df = pd.DataFrame(alignment_issues)

print("Total alignment issues:", len(alignment_df))

if not alignment_df.empty:
    print(
        "Case-only differences:",
        alignment_df["case_only"].sum()
    )

    print(
        "Other differences:",
        (~alignment_df["case_only"]).sum()
    )

    display(
        alignment_df[
            ["review_id", "pipeline", "case_only",
             "reference", "predicted"]
        ].head(10)
    )


Total alignment issues: 63
Case-only differences: 63
Other differences: 0


,review_id,pipeline,case_only,reference,predicted
0,R000929,B,True,Juicy,juicy
1,R001719,B,True,"Drinkisyummy,veryrecommend!","drinkisyummy,veryrecommend!"
2,R001967,B,True,"Loveit.Goodbuffet,fresh!","loveit.goodbuffet,fresh!"
3,R002484,B,True,Neveragain.Atmosphereishate!!,neveragain.atmosphereishate!!
4,R003897,B,True,"Cozydecor,cozy,atmosphereiscozy,amazinglunch!","cozydecor,cozy,atmosphereiscozy,amazinglunch!"
5,R004940,B,True,"Bad.Cakesalty,buffetsoggy!!","bad.cakesalty,buffetsoggy!!"
6,R005541,B,True,Soso.Veryfair!!,soso.veryfair!!
7,R007130,B,True,Niceplace.Veryrecommend!!,niceplace.veryrecommend!!
8,R008664,B,True,"Great.Noodlecozy,coffeeisfresh,premium,setfant...","great.noodlecozy,coffeeisfresh,premium,setfant..."
9,R015611,B,True,"Soso.Verynormal,fine,grillisaverage!","soso.verynormal,fine,grillisaverage!"


# **Cell 19 — Save Metrics**

In [22]:

# Cell 19: Save Evaluation Metrics

metrics_df.to_csv(
    "metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: metrics.csv")
display(metrics_df)


Saved: metrics.csv


,pipeline,engine,precision,recall,f1,TP,FP,FN
0,A,newmm,0.953262,0.927235,0.940069,64390,3157,5053
1,B,longest,0.945619,0.930245,0.937869,64599,3715,4844
2,C,attacut,0.923225,0.950679,0.936751,66018,5490,3425


# **Cell 20 — Save Paired Tokenization Outputs**

In [23]:

# Cell 20: Create Paired Outputs

import json

paired_rows = []

for _, row in frozen_eval.iterrows():
    text = row["normalized_text"]
    reference = str(row["tokens"]).split("|")

    item = {
        "review_id": row["review_id"],
        "normalized_text": text,
        "gold_tokens": json.dumps(reference, ensure_ascii=False)
    }

    for name, engine in pipelines.items():
        predicted = tokenize_text(text, engine)

        # Verify alignment
        ref_text = remove_spaces("".join(reference))
        pred_text = remove_spaces("".join(predicted))

        assert (
            len(ref_text) == len(pred_text)
            and ref_text.lower() == pred_text.lower()
        ), f"Alignment error: {row['review_id']}"

        item[f"tokens_{name}"] = json.dumps(
            predicted, ensure_ascii=False
        )

    paired_rows.append(item)

paired_df = pd.DataFrame(paired_rows)

paired_df.to_csv(
    "paired_outputs.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: paired_outputs.csv")
print("Total reviews:", len(paired_df))

display(paired_df.head(5))


Saved: paired_outputs.csv
Total reviews: 3000


,review_id,normalized_text,gold_tokens,tokens_A,tokens_B,tokens_C
0,R000088,ร้านนี้ดีจริง ผัดฉ่าเบคอนรสชาติเหมือนที่คุณแม่...,"[""ร้าน"", ""นี้"", ""ดี"", ""จริง"", ""ผัดฉ่า"", ""เบคอน...","[""ร้าน"", ""นี้"", ""ดีจริง"", ""ผัดฉ่า"", ""เบคอน"", ""...","[""ร้าน"", ""นี้"", ""ดีจริง"", ""ผัดฉ่า"", ""เบคอน"", ""...","[""ร้าน"", ""นี้"", ""ดี"", ""จริง"", ""ผัด"", ""ฉ่าเบคอน..."
1,R000098,มากินครั้งแรก สั่งตุ๋นสันคอหมูดอกแคมา โดนใจจริ...,"[""มา"", ""กิน"", ""ครั้ง"", ""แรก"", ""สั่ง"", ""ตุ๋น"", ...","[""มา"", ""กิน"", ""ครั้งแรก"", ""สั่ง"", ""ตุ๋น"", ""สัน...","[""มา"", ""กิน"", ""ครั้งแรก"", ""สั่ง"", ""ตุ๋น"", ""สัน...","[""มา"", ""กิน"", ""ครั้ง"", ""แรก"", ""สั่ง"", ""ตุ๋น"", ..."
2,R000115,พาเพื่อนร่วมงานมาฉลองปีใหม่ บัวลอยจานละ 410 บา...,"[""พา"", ""เพื่อนร่วมงาน"", ""มา"", ""ฉลอง"", ""ปีใหม่""...","[""พา"", ""เพื่อนร่วมงาน"", ""มา"", ""ฉลอง"", ""ปีใหม่""...","[""พา"", ""เพื่อนร่วมงาน"", ""มา"", ""ฉลอง"", ""ปีใหม่""...","[""พา"", ""เพื่อน"", ""ร่วม"", ""งาน"", ""มา"", ""ฉลอง"", ..."
3,R000128,แวะมาตอนไปอุดรธานี มาถึงได้โต๊ะเลย พูดตรง ๆ ผั...,"[""แวะ"", ""มา"", ""ตอน"", ""ไป"", ""อุดรธานี"", ""มา"", ""...","[""แวะ"", ""มา"", ""ตอน"", ""ไป"", ""อุดรธานี"", ""มาถึง""...","[""แวะ"", ""มา"", ""ตอน"", ""ไป"", ""อุดรธานี"", ""มาถึง""...","[""แวะ"", ""มา"", ""ตอน"", ""ไป"", ""อุดรธานี"", ""มา"", ""..."
4,R000135,ไม่ประทับใจเลย น้องมิ้นท์วางจานกระแทก แย่มาก ส...,"[""ไม่"", ""ประทับใจ"", ""เลย"", ""น้อง"", ""มิ้นท์"", ""...","[""ไม่"", ""ประทับใจ"", ""เลย"", ""น้อง"", ""มิ้นท์"", ""...","[""ไม่"", ""ประทับใจ"", ""เลย"", ""น้อง"", ""มิ้"", ""นท์...","[""ไม่"", ""ประทับใจ"", ""เลย"", ""น้องมิ้นท์"", ""วาง""..."


# **Cell 21 — Identify Tokenization Errors**

In [24]:

# Cell 21: Identify Tokenization Errors

error_rows = []

for _, row in paired_df.iterrows():

    reference = json.loads(row["gold_tokens"])
    ref_boundaries = get_boundaries(reference)

    for name, engine in pipelines.items():

        predicted = json.loads(row[f"tokens_{name}"])
        pred_boundaries = get_boundaries(predicted)

        false_positive = pred_boundaries - ref_boundaries
        false_negative = ref_boundaries - pred_boundaries

        if false_positive or false_negative:

            if false_positive and false_negative:
                error_type = "Mixed"
            elif false_positive:
                error_type = "Over-segmentation"
            else:
                error_type = "Under-segmentation"

            error_rows.append({
                "review_id": row["review_id"],
                "pipeline": name,
                "engine": engine,
                "text": row["normalized_text"],
                "gold_tokens": row["gold_tokens"],
                "predicted_tokens": row[f"tokens_{name}"],
                "FP": len(false_positive),
                "FN": len(false_negative),
                "error_type": error_type
            })

error_df = pd.DataFrame(error_rows)

print("Total error records:", len(error_df))

display(
    error_df.groupby(
        ["pipeline", "error_type"]
    ).size().unstack(fill_value=0)
)


Total error records: 7613


error_type,Mixed,Over-segmentation,Under-segmentation
pipeline,,,
A,1283,278,961
B,1183,302,1016
C,1444,808,338


# **Cell 22 — Select Five Error Examples**

In [25]:

# Cell 22: Select Five Error Examples

error_df["total_errors"] = (
    error_df["FP"] + error_df["FN"]
)

top_errors = (
    error_df.sort_values(
        "total_errors",
        ascending=False
    )
    .drop_duplicates(subset=["review_id"])
    .head(5)
    .copy()
)

top_errors["gold_tokens"] = top_errors[
    "gold_tokens"
].apply(lambda x: " | ".join(json.loads(x)))

top_errors["predicted_tokens"] = top_errors[
    "predicted_tokens"
].apply(lambda x: " | ".join(json.loads(x)))

display(
    top_errors[
        [
            "review_id",
            "pipeline",
            "engine",
            "text",
            "gold_tokens",
            "predicted_tokens",
            "error_type",
            "FP",
            "FN"
        ]
    ]
)

top_errors.to_csv(
    "error_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: error_analysis.csv")


,review_id,pipeline,engine,text,gold_tokens,predicted_tokens,error_type,FP,FN
5535,R071192,A,newmm,ข้าวแกงบุศราแถวอำนาจเจริญ แล้วก็แนะนำผัดพริกเก...,ข้าว | แกง | บุศรา | แถว | อำนาจเจริญ | แล้ว |...,ข้าวแกง | บุ | ศ | รา | แถว | อำนาจเจริญ | แล้...,Mixed,10,8
1637,R020229,B,longest,มากินครั้งแรก ขอบอกว่าทอดกระเทียมหอยเชลล์เหม็น...,มา | กิน | ครั้ง | แรก | ขอ | บอก | ว่า | ทอด ...,มา | กิน | ครั้งแรก | ขอบ | อก | ว่า | ทอด | ก...,Mixed,8,9
1866,R023210,C,attacut,มากินครั้งแรก เจลาโต้มันม่วงหวานกำลังดีมาก ผู้...,มา | กิน | ครั้ง | แรก | เจลาโต้ | มันม่วง | ห...,มา | กิน | ครั้ง | แรก | เจลา | โต้ | มัน | ม่...,Mixed,11,6
3717,R047705,A,newmm,ขับรถมาจากอัมพวาเพื่อร้านนี้ คุ้มค่ากับราคา กล...,ขับ | รถ | มา | จาก | อัมพวา | เพื่อ | ร้าน | ...,ขับรถ | มาจาก | อัมพวา | เพื่อ | ร้าน | นี้ | ...,Mixed,9,8
6750,R088301,B,longest,น้าบุษกรแถวลำพูน ผัดพริกเกลือกุนเชียงรสชาตไิม่...,น้า | บุษกร | แถว | ลำพูน | ผัด | พริก | เกลือ...,น้า | บุษกร | แถว | ลำพูน | ผัด | พริก | เกลือ...,Mixed,15,2


Saved: error_analysis.csv


# **Cell 23 — ปรับปรุง Error Analysis**

In [26]:

# Cell 23: Select Diverse Error Examples

# Select examples from different pipelines and error types
selections = [
    ("A", "Under-segmentation"),
    ("A", "Mixed"),
    ("B", "Under-segmentation"),
    ("B", "Mixed"),
    ("C", "Over-segmentation"),
    ("C", "Mixed")
]

selected_rows = []

for pipeline, error_type in selections:
    subset = error_df[
        (error_df["pipeline"] == pipeline) &
        (error_df["error_type"] == error_type)
    ].sort_values(
        "total_errors", ascending=False
    )

    if not subset.empty:
        selected_rows.append(subset.iloc[0].copy())

diverse_errors = pd.DataFrame(selected_rows)

# Make tokens readable
for col in ["gold_tokens", "predicted_tokens"]:
    diverse_errors[col] = diverse_errors[col].apply(
        lambda x: " | ".join(json.loads(x))
    )

display(
    diverse_errors[
        [
            "review_id",
            "pipeline",
            "engine",
            "gold_tokens",
            "predicted_tokens",
            "error_type",
            "FP",
            "FN"
        ]
    ]
)

diverse_errors.to_csv(
    "error_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: error_analysis.csv")
print("Selected examples:", len(diverse_errors))


,review_id,pipeline,engine,gold_tokens,predicted_tokens,error_type,FP,FN
4046,R052231,A,newmm,ตาม | รีวิว | จาก | เพจ | รีวิว | มา | ไม่ | ผ...,ตาม | รีวิว | จาก | เพจ | รีวิว | มา | ไม่ | ผ...,Under-segmentation,0,8
5535,R071192,A,newmm,ข้าว | แกง | บุศรา | แถว | อำนาจเจริญ | แล้ว |...,ข้าวแกง | บุ | ศ | รา | แถว | อำนาจเจริญ | แล้...,Mixed,10,8
4047,R052231,B,longest,ตาม | รีวิว | จาก | เพจ | รีวิว | มา | ไม่ | ผ...,ตาม | รีวิว | จาก | เพจ | รีวิว | มา | ไม่ | ผ...,Under-segmentation,0,8
1637,R020229,B,longest,มา | กิน | ครั้ง | แรก | ขอ | บอก | ว่า | ทอด ...,มา | กิน | ครั้งแรก | ขอบ | อก | ว่า | ทอด | ก...,Mixed,8,9
4616,R059465,C,attacut,มา | กับ | รุ่นน้อง | แบบ | ว่า | ซาชิมิ | ไม่...,มา | กับ | รุ่น | น้อง | แบบ | ว่า | ซาชิ | มิ...,Over-segmentation,8,0
1866,R023210,C,attacut,มา | กิน | ครั้ง | แรก | เจลาโต้ | มันม่วง | ห...,มา | กิน | ครั้ง | แรก | เจลา | โต้ | มัน | ม่...,Mixed,11,6


Saved: error_analysis.csv
Selected examples: 6


# **Cell 24 — Create preprocess.py**

In [27]:

%%writefile preprocess.py

import re
import unicodedata
from pythainlp.tokenize import word_tokenize


def normalize_minimal(text):
    """Minimal text normalization."""
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"[\u200B\u200C\u200D\uFEFF]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_text(text, engine="newmm"):
    """Tokenize text using a selected engine."""
    return word_tokenize(
        text,
        engine=engine,
        keep_whitespace=False
    )


def remove_spaces(text):
    """Remove whitespace for boundary evaluation."""
    return "".join(str(text).split())


def get_boundaries(tokens):
    """Return token boundary positions."""
    boundaries = set()
    position = 0

    for token in tokens:
        token = remove_spaces(token)
        position += len(token)
        boundaries.add(position)

    boundaries.discard(position)
    return boundaries


Writing preprocess.py


# **Cell 25 — Test preprocess.py**

In [28]:

# Cell 25: Test preprocess.py

import preprocess

text = "อาหาร   อร่อยมาก"

normalized = preprocess.normalize_minimal(text)
tokens = preprocess.tokenize_text(
    normalized, engine="newmm"
)

print("Normalized:", normalized)
print("Tokens:", tokens)
print("Boundaries:", preprocess.get_boundaries(tokens))


Normalized: อาหาร อร่อยมาก
Tokens: ['อาหาร', 'อร่อย', 'มาก']
Boundaries: {10, 5}


# **Cell 26 — Download Lab Files**

In [29]:

# Cell 26: Download Lab Files

import os
import zipfile
from google.colab import files

required_files = [
    "sampled_20000.csv",
    "frozen_eval.csv",
    "metrics.csv",
    "paired_outputs.csv",
    "error_analysis.csv",
    "preprocess.py"
]

zip_name = "Lab2_1_Results.zip"

with zipfile.ZipFile(zip_name, "w") as zipf:
    for filename in required_files:
        if os.path.exists(filename):
            zipf.write(filename)
            print("Added:", filename)
        else:
            print("Missing:", filename)

files.download(zip_name)


Added: sampled_20000.csv
Added: frozen_eval.csv
Added: metrics.csv
Added: paired_outputs.csv
Added: error_analysis.csv
Added: preprocess.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Cell 27 — Evaluate Special Cases**

In [30]:

# Cell 27: Evaluate Negation and Mixed-language Cases

import re
import json
import pandas as pd

# Select reviews by case
negation_df = paired_df[
    paired_df["normalized_text"].str.contains("ไม่", regex=False)
].copy()

mixed_df = paired_df[
    paired_df["normalized_text"].str.contains(r"[ก-๙]", regex=True) &
    paired_df["normalized_text"].str.contains(r"[A-Za-z]", regex=True)
].copy()

def evaluate_case(data, case_name):
    rows = []

    for name, engine in pipelines.items():
        total_tp = total_fp = total_fn = 0

        for _, row in data.iterrows():
            reference = json.loads(row["gold_tokens"])
            predicted = json.loads(row[f"tokens_{name}"])

            _, _, _, tp, fp, fn = calculate_boundary_metrics(
                reference, predicted
            )

            total_tp += tp
            total_fp += fp
            total_fn += fn

        precision = (
            total_tp / (total_tp + total_fp)
            if total_tp + total_fp else 0
        )

        recall = (
            total_tp / (total_tp + total_fn)
            if total_tp + total_fn else 0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        rows.append({
            "case": case_name,
            "pipeline": name,
            "engine": engine,
            "reviews": len(data),
            "precision": round(precision * 100, 2),
            "recall": round(recall * 100, 2),
            "f1": round(f1 * 100, 2)
        })

    return rows


results = (
    evaluate_case(negation_df, "Negation") +
    evaluate_case(mixed_df, "Mixed-language")
)

special_metrics_df = pd.DataFrame(results)

display(special_metrics_df)


,case,pipeline,engine,reviews,precision,recall,f1
0,Negation,A,newmm,1254,95.53,92.77,94.13
1,Negation,B,longest,1254,94.85,93.04,93.94
2,Negation,C,attacut,1254,91.95,95.48,93.68
3,Mixed-language,A,newmm,321,95.40,93.39,94.39
4,Mixed-language,B,longest,321,90.43,93.75,92.06
5,Mixed-language,C,attacut,321,92.00,95.81,93.87
